# 03 — Fine-Tune and Merge

[Notebook guide](README.md) · [Project overview](../README.md) · [English paper](../docs/en/paper.pdf)

Run QLoRA adaptation and merge the adapters with the base model.

| Item | Details |
| --- | --- |
| Inputs | Combined JSONL dataset; access to Llama 3.1 8B Instruct; optional saved checkpoints. |
| Outputs | Training checkpoints, merged model files, and an inference check. |
| Requirements | Google Colab/Drive, GPU runtime (recorded setup: T4), gated-model access, and sufficient CPU RAM and disk space for merging. |
| Configuration overview | Review `data_path`, `model_id`, `training_args.output_dir`, `adapter_path`, merge `output_dir`, `final_drive_path`, and `model_path`. NF4 4-bit training; rank 16; alpha 32; one epoch; max length 1024. |

**Recorded experiment:** run code cells in their existing order. This presentation edit preserves executable code, parameters, prompts, data, and experimental outputs. Configuration is summarized here without moving assignments. Installation logs are omitted; execution counts belong to the original run. Serbian prompts, reference text, docstrings, and printed outputs remain verbatim.


## Workflow

Load the model and tokenizer, configure quantization and adapters, format examples, train, merge adapter weights, and inspect an inference example.

## 1. Install dependencies

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install trl peft

In [ ]:
!pip install -U "torchao>=0.16.0"

## 2. Import training libraries

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import drive
from datasets import load_dataset
from transformers.trainer_utils import get_last_checkpoint
from peft import prepare_model_for_kbit_training
from peft import PeftModel
from peft import AutoPeftModelForCausalLM


### Additional imports for merging

In [ ]:
from huggingface_hub import login
from google.colab import drive




## 3. Mount Drive and load training data

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Dataset path on Drive; adjust if stored in another directory
data_path = "/content/drive/MyDrive/NLP_proyect/datasets/final_train_dataset.json"

train_dataset = load_dataset("json", data_files=data_path, split="train")

NameError: name 'load_dataset' is not defined

### Hugging Face authentication

In [ ]:
HF_TOKEN = ""

login(token=HF_TOKEN)

## 4. Load the base model and tokenizer

In [ ]:
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# 1. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 2. Configure 4-bit quantization (BitsAndBytes)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 3. Load the model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Model i tokenizator su uspešno učitani!")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model i tokenizator su uspešno učitani!


## 5. Prepare the model and configure LoRA

In [ ]:
# Set the internal dtype configuration to float16
model.config.torch_dtype = torch.float16

# Prepare the model for k-bit training
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

## 6. Format training prompts

The original Serbian instruction template and end-of-sequence token are preserved.

In [ ]:
def create_prompt(row):
    if row["input"]:
        text = f"### Instrukcija:\n{row['instruction']}\n\n### Ulaz:\n{row['input']}\n\n### Odgovor:\n{row['output']}"
    else:
        text = f"### Instrukcija:\n{row['instruction']}\n\n### Odgovor:\n{row['output']}"

    return {"text": text + tokenizer.eos_token}

train_dataset = train_dataset.map(create_prompt)

Map:   0%|          | 0/3152 [00:00<?, ? examples/s]

In [ ]:
print(train_dataset['text'][0])

### Instrukcija:
Koji je najduži suspendirani most u Sjedinjenim Državama?

### Ulaz:
Most Džordž Vašington je najprometniji most za motorna vozila na svetu, koji povezuje Manhatten sa okrugom Bergen, u Nju Džersiju. Verazano-Narouzski most je najduži suspendirani most u Americi i jedan od najdužih na svetu. Bruklinski most je ikona grada. Kule Bruklinskog mosta sagrađene su od varoviča, granita i Rozendejl cementa, a njihov arhitektonski stil je neo-gotički, sa karakterističnim oštrim lukovima iznad prolaza kroz kamene kule. Ovaj most je takođe bio i najduži suspendioni most na svetu od svog otvaranja do 1903. godine, i prvi je suspendioni most sa čeličnim žicom.

### Odgovor:
Verazano-Narouzski most<|eot_id|>


## 7. Configure training

In [ ]:
training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/NLP_proyect/outputs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=False,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    dataset_text_field="text",  # Configured here
    max_length=1024         # Configured here
)

### Initialize the trainer

In [ ]:
# Explicitly convert any remaining bfloat16 tensors
for param in model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args
)

Adding EOS to train dataset:   0%|          | 0/3152 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3152 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3152 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3152 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3152 [00:00<?, ? examples/s]

## 8. Run or resume fine-tuning

The next cell resumes from the last checkpoint when one exists. Saved training results below belong to the original run.

In [ ]:
output_dir = "/content/drive/MyDrive/NLP_proyect/outputs"
last_checkpoint = get_last_checkpoint(output_dir)

if last_checkpoint:
    print(f"Nastavljam obuku od: {last_checkpoint}")
else:
    print("Započinjem novu obuku.")

trainer.train(resume_from_checkpoint=last_checkpoint)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Nastavljam obuku od: /content/drive/MyDrive/NLP_proyect/outputs/checkpoint-200


Step,Training Loss
210,1.343760
220,1.232341
230,1.327529
240,1.320329
250,1.305310
260,1.398979


Step,Training Loss
210,1.343760
220,1.232341
230,1.327529
240,1.320329
250,1.305310
260,1.398979
270,1.420551
280,1.329464
290,1.291755
300,1.340667


TrainOutput(global_step=394, training_loss=0.655698030733215, metrics={'train_runtime': 13480.9517, 'train_samples_per_second': 0.234, 'train_steps_per_second': 0.029, 'total_flos': 4.721407984292659e+16, 'train_loss': 0.655698030733215, 'entropy': 1.3742114920169115, 'num_tokens': 514256.0, 'mean_token_accuracy': 0.7042398359626532, 'epoch': 1.0})

## 9. Merge adapters into base-model shards

The CPU procedure processes shards individually, then saves supporting files and copies the merged model to Drive.

In [ ]:
import os, gc, json, shutil
import torch
from pathlib import Path
from safetensors import safe_open
from safetensors.torch import save_file
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

In [ ]:
# ---------- CONFIGURATION ----------
base_model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
adapter_path = "/content/drive/MyDrive/NLP_proyect/outputs/checkpoint-394"
output_dir = "/content/merged_model_local"
target_dtype = torch.float16
# -----------------------------

os.makedirs(output_dir, exist_ok=True)

print("Tražim lokalni keš baznog modela...")
base_dir = Path(snapshot_download(base_model_id))

Tražim lokalni keš baznog modela...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

In [ ]:
print("Učitavam LoRA adapter...")
with open(os.path.join(adapter_path, "adapter_config.json")) as f:
    adapter_config = json.load(f)

r = adapter_config["r"]
lora_alpha = adapter_config["lora_alpha"]
use_rslora = adapter_config.get("use_rslora", False)
use_dora = adapter_config.get("use_dora", False)

if use_dora:
    raise ValueError("Adapter koristi DoRA – ova skripta ne pokriva taj slučaj.")

scaling = lora_alpha / (r ** 0.5) if use_rslora else lora_alpha / r
print(f"r={r}, lora_alpha={lora_alpha}, scaling={scaling:.4f}")

adapter_weights_path = os.path.join(adapter_path, "adapter_model.safetensors")
lora_state = {}
with safe_open(adapter_weights_path, framework="pt", device="cpu") as f:
    for key in f.keys():
        lora_state[key] = f.get_tensor(key)

lora_map = {}
prefix = "base_model.model."
for key, tensor in lora_state.items():
    if not key.startswith(prefix):
        continue
    if ".lora_A." in key:
        base_name = key.replace(prefix, "").split(".lora_A.")[0] + ".weight"
        lora_map.setdefault(base_name, {})["A"] = tensor.float()
    elif ".lora_B." in key:
        base_name = key.replace(prefix, "").split(".lora_B.")[0] + ".weight"
        lora_map.setdefault(base_name, {})["B"] = tensor.float()

print(f"Pronađeno {len(lora_map)} slojeva sa LoRA adapterom.")
del lora_state
gc.collect()


Učitavam LoRA adapter...
r=16, lora_alpha=32, scaling=2.0000
Pronađeno 224 slojeva sa LoRA adapterom.


171

In [ ]:
# --- 2. Base-model shard index ---
index_path = base_dir / "model.safetensors.index.json"
with open(index_path) as f:
    index = json.load(f)

weight_map = index["weight_map"]
shards = {}
for tensor_name, shard_file in weight_map.items():
    shards.setdefault(shard_file, []).append(tensor_name)

print(f"Bazni model: {len(shards)} shard-ova, {len(weight_map)} tenzora.")


Bazni model: 4 shard-ova, 291 tenzora.


In [ ]:
# --- 3. Merge one shard at a time ---
merged_count = 0
for shard_file, tensor_names in shards.items():
    print(f"  Obrađujem {shard_file} ({len(tensor_names)} tenzora)...")
    shard_path = base_dir / shard_file
    out_tensors = {}

    with safe_open(shard_path, framework="pt", device="cpu") as f:
        for name in tensor_names:
            w = f.get_tensor(name)

            if name in lora_map and "A" in lora_map[name] and "B" in lora_map[name]:
                A, B = lora_map[name]["A"], lora_map[name]["B"]
                delta = (B @ A) * scaling
                merged = (w.float() + delta).to(target_dtype).contiguous()
                out_tensors[name] = merged
                merged_count += 1
                del delta, merged
            else:
                out_tensors[name] = w.to(target_dtype).contiguous() if w.dtype != target_dtype else w.contiguous()
            del w

    save_file(out_tensors, os.path.join(output_dir, shard_file), metadata={"format": "pt"})
    del out_tensors
    gc.collect()

print(f"Spojeno {merged_count} / {len(lora_map)} slojeva iz adaptera.")
if merged_count != len(lora_map):
    print("⚠️ UPOZORENJE: broj spojenih slojeva se ne poklapa sa brojem u adapteru — proveri imenovanje ključeva!")


  Obrađujem model-00004-of-00004.safetensors (5 tenzora)...
  Obrađujem model-00001-of-00004.safetensors (82 tenzora)...
  Obrađujem model-00002-of-00004.safetensors (104 tenzora)...
  Obrađujem model-00003-of-00004.safetensors (100 tenzora)...
Spojeno 224 / 224 slojeva iz adaptera.


In [ ]:
# --- 4. Supporting files ---
shutil.copy(index_path, os.path.join(output_dir, "model.safetensors.index.json"))
for fname in ["config.json", "generation_config.json"]:
    src = base_dir / fname
    if src.exists():
        shutil.copy(src, os.path.join(output_dir, fname))

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.save_pretrained(output_dir)

print("Gotovo! Spojen model je u:", output_dir)

Gotovo! Spojen model je u: /content/merged_model_local


In [ ]:
import shutil

final_drive_path = "/content/drive/MyDrive/NLP_proyect/merged_model"

print("Kopiram na Google Drive (ovo može potrajati nekoliko minuta za ~16GB)...")
shutil.copytree(output_dir, final_drive_path, dirs_exist_ok=True)
print("Gotovo! Model je na:", final_drive_path)

Kopiram na Google Drive (ovo može potrajati nekoliko minuta za ~16GB)...
Gotovo! Model je na: /content/drive/MyDrive/NLP_proyect/merged_model


## 10. Inspect merged-model inference

This is the recorded inference check; its input and output have not been edited.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "/content/drive/MyDrive/NLP_proyect/merged_model"

print("Učitavam tokenizer i model...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

prompt = "Zdravo! Kako funkcioniše obrada prirodnog jezika?"
messages = [{"role": "user", "content": prompt}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,   # Use the configured tokenizer
).to("cuda")

print("Generišem odgovor...")
outputs = model.generate(
    **inputs,           # Unpack the dictionary into keyword arguments
    max_new_tokens=150,
    temperature=0.7,
    do_sample=True,
)

odgovor = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n--- IZLAZ MODELA ---")
print(odgovor)

Učitavam tokenizer i model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Generišem odgovor...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



--- IZLAZ MODELA ---
system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Zdravo! Kako funkcioniše obrada prirodnog jezika?assistant

Zdravo! Obrada prirodnog jezika (NLP) je oblast računarske nauke koja se bavi razvojem algoritama i modela za obradu i analizu prirodnog jezika. Cilj NLP-a je da kompjuteri mogu da razumeju, generišu i interagiraju sa ljudskim jezikom na način koji je sličan ljudskom ponašanju.

Postoje nekoliko glavnih komponenti NLP-a:

1. **Leksikografija**: Ovaj segment se bavi analizom i generišuđem reči i fra


## Reading the recorded results

Saved outputs document the original run. This presentation edit does not rerun the notebook, change its artifacts, or claim that a fresh environment will reproduce the recorded results exactly.